In [ ]:
import pickle
import time
import tiktoken
import datetime
import json
import requests
import traceback
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from jsonschema import validate
from openai import OpenAI, RateLimitError, APITimeoutError, InternalServerError, Timeout
from tenacity import retry, stop_after_attempt, wait_incrementing, retry_if_exception_type, after_log, before_sleep_log
import logging
from IPython.display import clear_output
from collections import defaultdict
from multiprocessing import Process, Manager
from collections import defaultdict

In [ ]:
# enter your OpenRouter API key below
OPENROUTER_API_KEY = 'sk-...' 

In [ ]:
request_limit_per_minute = 500
token_limit_per_minute = 2e6

request_timeout_seconds = 120   # maximum wait time for openAI to respond before triggering request timeout 
request_max_retries = 1         # number to times to automatically retry failed requests
tpm_wait_polling_seconds = 10    # if our internal TPM estimate thinks TPM limit is exceeded, how often to check if limit cleared

# global logger for static classes
logger = logging.getLogger()
logging.basicConfig(level=logging.INFO)

# Helper Class for Calling LLMs

In [ ]:
class OpenRouter:
    def __init__(self, model_provider_order,
                 halt_on_error=True,
                 is_verbose=True,
                 timeout=request_timeout_seconds,
                 max_retries=request_max_retries,
                 request_limit_per_minute=request_limit_per_minute,
                 token_limit_per_minute=token_limit_per_minute,
                 tpm_wait_polling_seconds=tpm_wait_polling_seconds,
                 logger=logger,
                 api_key=OPENROUTER_API_KEY):
        self.model, self.provider_order, self.model_canonical_name = model_provider_order
        self.halt_on_error = halt_on_error
        self.is_verbose = is_verbose
        self.tpm_wait_polling_seconds = tpm_wait_polling_seconds
        self.request_limit_per_minute = request_limit_per_minute
        self.request_delay_seconds = 60.0 / request_limit_per_minute
        self.token_limit_per_minute = token_limit_per_minute
        self.response_history = []
        self.message_history = {}
        self.logger = logger
        likert_options = [
            "Very Inaccurate",
            "Moderately Inaccurate",
            "Neither Accurate nor Inaccurate",
            "Moderately Accurate",
            "Very Accurate",
        ]
        # Sort by length to match longer options (e.g., "Strongly Agree") before shorter ones (e.g., "Agree")
        self.likert_options = sorted(likert_options, key=len, reverse=True)
        self.default_seed = 1 if 'x-ai' in self.model else 0
        
        self.client = OpenAI(base_url="https://openrouter.ai/api/v1",
                             api_key = api_key,
                             timeout=timeout,
                             max_retries=max_retries)

        
    def extract_float_response(self, content):
        # Search for a float or integer between 0 and 100
        match = re.search(r'\b(100(?:\.0+)?|[0-9]?[0-9](?:\.\d+)?|\d{1,2})\b', content)
        if match:
            value = float(match.group(0))
            if 0.0 <= value <= 100.0:
                return json.dumps({"response": value})
        raise Exception("No valid float response found in: ", content)
    
    
    def get_running_cost_num_prompt_completion_tokens(self):
        """
        This function computes the total cost (estimated) of all
        messages sent by the instance of LLM called from
        Returns: total_running_cost, total_num_prompt_tokens, total_num_response_tokens
        """
        n_prompt_tokens = np.sum([x.usage.prompt_tokens for x in self.response_history])
        n_completion_tokens = np.sum([x.usage.completion_tokens for x in self.response_history])
        total_cost = sum(r.usage.cost for r in self.response_history)
        return (total_cost,
                n_prompt_tokens,
                n_completion_tokens)

    def get_key_usage_credits(self):
        # get what OpenRouter says the api key has used in total
        # returns usage, total credits available
        resp = requests.get(
            "https://openrouter.ai/api/v1/credits",
            headers={"Authorization": f"Bearer {self.client.api_key}"}
        )
        resp.raise_for_status()
        info = resp.json()["data"]
        return info["total_usage"], info["total_credits"]

    # retry failing requests starting with 10 second wait,
    # increasing wait time by 10 seconds each retry, up to a max window of 120s (or 5 times)
    # the goal is to try to avoid hitting backoff,
    # we treat this as a last resort because of its runtime cost
    @retry(wait=wait_incrementing(start=10, increment=10, max=120),
           stop=stop_after_attempt(5),
           retry=retry_if_exception_type((RateLimitError, APITimeoutError, InternalServerError, Timeout)),
           before_sleep=before_sleep_log(logger, logging.INFO),
           after=after_log(logger, logging.INFO))
    def completion_with_backoff(self, client, **kwargs):
        return client.chat.completions.create(**kwargs)


    def check_internal_TPM_tracker(self, n_message_tokens):
        """
        Checks internal TPM count to see if a message with length = n_message_tokens
        can be sent. If not, it waits (sleeps - blocking) until the message delivery
        meets into TPM limit
        """
        now = datetime.datetime.now()
        one_minute_ago = now + datetime.timedelta(seconds=-60)
        self.token_count_history = [x for x in self.token_count_history if x[1] > one_minute_ago]
        n_tokens_past_minute = np.sum([x[0] for x in self.token_count_history]) + n_message_tokens
        # fixed delay waiting if TPM exceeded over past minute
        # this is cpu polling, so it doesnt cost money or much compute
        while n_tokens_past_minute > self.token_limit_per_minute:
            if self.is_verbose: self.logger.info(f'Internal TPM limit exceeded, waiting for {self.tpm_wait_polling_seconds} seconds...')
            time.sleep(self.tpm_wait_polling_seconds)
            now = datetime.datetime.now()
            one_minute_ago = now + datetime.timedelta(seconds=-60)
            self.token_count_history = [x for x in self.token_count_history if x[1] > one_minute_ago]
            n_tokens_past_minute = np.sum([x[0] for x in self.token_count_history]) + n_message_tokens
        now = datetime.datetime.now()
        self.token_count_history.append((n_message_tokens, now))


    def send_message(self, system_role, message, json_schema, validate_response=True):
        """
        This is the primary function used to send messages to GPT and get responses
        Steps are:
          - check that json schema meets our basic requirements
          - handle RPM and TPM limits as best as we can
            (when openai rejects requests for exceeding limits its much slower)
          - build and send the message using openai ChatCompletion api
          - perform basic validation on GPT's response
        This function either returns a ChatCompletion response object or None (if failure occurred)
        Errors are propogated using raised Exceptions
        """
        # sleep based on RPM limit (lazy logic, avoids keeping running count of actual requests per minute)
        time.sleep(self.request_delay_seconds)

        # check TPM limit (not lazy, uses running count of tokens per minute)
        try:
            encoding = tiktoken.encoding_for_model(self.model)
        except:
            encoding = tiktoken.encoding_for_model('gpt-4')
        n_message_tokens = len(encoding.encode(system_role)) + len(encoding.encode(message))
        self.logger.info(f'processing message with {n_message_tokens} tokens...')
        if n_message_tokens > self.token_limit_per_minute:
            return self.bad_response_output(f'Unable to send message as it exceeds TPM. Number of tokens in message = {n_message_tokens}')
        
        # build and send message over openai-api
        message_id = len(self.message_history.keys())
        self.message_history[message_id] = [] if 'x-ai' in self.model else [{"role": "system", "content": system_role}]
        self.message_history[message_id].append({"role": "user", "content": message})
        try:
            response = self.completion_with_backoff(self.client,
                                                    model=self.model,
                                                    messages=self.message_history[message_id],
                                                    temperature=0,
                                                    stream=False,
                                                    extra_body={"usage": {"include": True},
                                                                "reasoning": {# One of the following (not both):
                                                                              "effort": "medium", # Can be "high", "medium", or "low" (OpenAI-style)
                                                                              # Optional: Default is false. All models support this.
                                                                              "exclude": False # Set to true to exclude reasoning tokens from response
                                                                              },
                                                                "provider": {"order": self.provider_order, 
                                                                             "sort": "price",
                                                                             "data_collection": "deny",
                                                                             "allow_fallbacks": False}},
                                                    response_format={"type": "json_schema",
                                                                     "json_schema": json_schema},
                                                    seed=self.default_seed, logprobs=False)
            
            self.response_history.append(response)
            # reasoning models dont return a content field
            if response.choices[0].message.content is None:
                self.bad_response_output(f'None in message content')
                return None
            elif response.choices[0].message.content == '':
                if hasattr(response.choices[0].message, 'reasoning'):
                    if response.choices[0].message.reasoning != '':
                        response.choices[0].message.content = response.choices[0].message.reasoning
            else:
                pass
        except Exception as e:
            if self.halt_on_error:
                raise
            else:
                if self.is_verbose:
                    str_e = str(e)
                    self.logger.info(f'An exception occurred: {str_e}')
                    self.logger.info(traceback.format_exc())
                return None

        return (response, message_id)
        

    def bad_response_output(self, error):
        # general function for informing the user when an error occurs
        if self.halt_on_error:
            raise Exception(error)
        else:
            if self.is_verbose:
                self.logger.info(f'Error - {error}')
        return None

# Load AAPECS Data

In [ ]:
DATA_ROOT = './Data'  # please contact the authors for access to the data
pheno_df, sub_transcripts, sub_dpds = pickle.load(open(f'{DATA_ROOT}/AAPECS.pickle', 'rb'))

In [ ]:
questions_list = [
    {'text': 'anxiety, guilt, shame, irritability, worry, emotional volatility, fear, tendency to experience distressing emotions intensely, or self-criticism','domain':'Negative Affectivity','qid':'dpdsNA'},
    {'text': 'arrogance, hostility, callousness, manipulation, entitlement, grandiosity, interpersonal conflict, exploitation, or lack of empathy','domain':'Antagonism','qid':'dpdsAN'},
    {'text': 'social withdrawal, emotional numbness, emotional distance, restricted emotional expression, anhedonia, or a lack of or avoidance of intimacy or connection','domain':'Detachment','qid':'dpdsDET'},
    {'text': 'impulsivity, irresponsibility, recklessness, risk taking, poor planning, disregard for consequences, or difficulty delaying gratification','domain':'Disinhibition','qid':'dpdsDI'},
    {'text': 'perfectionism, rigid control, preoccupation with rules or order, fear of mistakes, or over-conscientiousness','domain':'Anankastia','qid':'dpdsAK'},
]

system_role = ''

# Prepare Prompt Template

In [ ]:
def get_prompt(thoughts, domain, signs):
    prompt_template = f"""
Your task is to assess the participant's level of {domain} based on their daily diary entry describing the most significant event that occurred during their day. 

Respond as if you are the participant, using their emotional tone and inner reactions to guide your answer.

Pay attention to signs of {signs}. Use contextual reasoning to determine how strongly the participant likely experienced or expressed {domain}.

### Data
Participant's daily diaries:
{thoughts}

### Expected Output
Respond with a single number from 0 to 100, where:
- 0 means "not at all"
- 100 means "very much"

Then:
- Provide a list of justifications that support your ratings. Each justification must include:
  - A brief explanation of a theme or observation that informed your judgment.
  - One or more direct quotes or excerpts from the diary that illustrate or support that explanation.
  - For each quote, include the **exact diary date** from which it was drawn.
  - Write the justifications in the rater's voice (third-person), not as the participant.

The quotes should be integrated into the reasoning — each segment should combine narrative interpretation and evidence.
"""
    return prompt_template

In [ ]:
def find_required_fields(schema, parent_key=''):
    required_fields = []
    if 'required' in schema:
        # If parent_key exists, prefix it to the required field names
        for field in schema['required']:
            full_field_name = f"{parent_key}.{field}" if parent_key else field
            required_fields.append(full_field_name)

    if 'properties' in schema:
        for key, value in schema['properties'].items():
            new_parent_key = f"{parent_key}.{key}" if parent_key else key
            required_fields.extend(find_required_fields(value, new_parent_key))

    return required_fields


def get_value_from_path(data, path):
    keys = path.split('.')
    for key in keys:
        if isinstance(data, list):
            key = int(key)
        data = data[key]
    return data


def get_required_values(schema, response):
    required_fields = find_required_fields(schema)
    required_values = {}

    for field in required_fields:
        try:
            value = get_value_from_path(response, field)
            required_values[field] = value
        except KeyError:
            required_values[field] = None  # Handle missing values if needed

    return required_values

In [ ]:
def generic_messaging_wrapper(chat, system_role, message, json_schema):
    # with halt_on_error set to True in OpenRouter class, 
    # we use exception propogation to handle errors and edge-cases
    message_history_id = -1
    required_values = None
    try:
        response, message_history_id = chat.send_message(system_role=system_role,
                                                         message=message, 
                                                         json_schema=json_schema)
        assert response is not None
        response_json = json.loads(response.choices[0].message.content)
        required_values = response_json
    except Exception as e:
        response_json = {}
        chat.logger.info(f'Messaging wrapper failure - {str(e)}')
        print(traceback.format_exc())

    cost, n_prompt_tokens, n_completion_tokens = chat.get_running_cost_num_prompt_completion_tokens()
    return required_values, cost

In [ ]:
def make_dpds_domain_json_schema(domain):
    """
    Schema for a single DPDS/trait-domain rating (0–100, continuous)
    from daily diaries with evidence-based justifications (dated quotes).
    """
    safe_name = domain.replace(' ', '_').lower()

    return {
        "name": f"{safe_name}_assessment_from_daily_diaries",
        "description": f"Assesses the participant's level of {domain} based on daily diary entries.",
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "strict": True,
            "properties": {
                "rating_0_100": {
                    "type": "number",
                    "minimum": 0.0,
                    "maximum": 100.0,
                    "description": (
                        f"Continuous rating from 0.0 to 100.0 indicating how strongly "
                        f"the diary evidence reflects {domain}."
                    )
                },
                "justifications": {
                    "type": "array",
                    "minItems": 1,
                    "description": (
                        "Justification segments integrating interpretation and verbatim "
                        "diary evidence with dates."
                    ),
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "explanation": {
                                "type": "string",
                                "minLength": 1,
                                "description": (
                                    f"Brief theme/observation relevant to {domain} that "
                                    "informed the rating."
                                )
                            },
                            "quotes": {
                                "type": "array",
                                "minItems": 1,
                                "description": (
                                    "Direct quotes/excerpts from the diary supporting the explanation, "
                                    "each with the exact diary date."
                                ),
                                "items": {
                                    "type": "object",
                                    "additionalProperties": False,
                                    "properties": {
                                        "text": {
                                            "type": "string",
                                            "minLength": 1,
                                            "description": "Verbatim quote/excerpt from the diary."
                                        },
                                        "date": {
                                            "type": "string",
                                            "format": "date",
                                            "description": "Exact diary date (YYYY-MM-DD) for the quote."
                                        }
                                    },
                                    "required": ["text", "date"]
                                }
                            }
                        },
                        "required": ["explanation", "quotes"]
                    }
                }
            },
            "required": [
                "rating_0_100",
                "justifications"
            ]
        }
    }


# Helper Functions For Data Wrangling

In [ ]:
def format_dpds_domain_summary(data, domain, subj, true_score=None, model_name="GPT"):
    """
    Formats a human-readable summary for a single DPDS domain assessment.
    """
    rating = np.round(data["rating_0_100"], 2)

    lines = []
    sep = "─" * 72

    # Header
    lines.append(sep)
    lines.append(f"Subject#{subj}: {domain}")
    lines.append(sep)

    if true_score is not None:
        true_score = np.round(true_score, 2)
        lines.append(f"Self-Reported Score : {true_score}")

    lines.append(f"{model_name} Rating          : {rating}")
    lines.append("")

    # Justifications
    for i, entry in enumerate(data["justifications"], 1):
        lines.append(f"▸ Reason {i}")
        lines.append(f"  {entry['explanation']}")

        quotes = entry.get("quotes", [])
        if quotes:
            label = "Citation" if len(quotes) == 1 else "Citations"
            lines.append(f"  {label}:")
            for quote in quotes:
                lines.append(
                    f"    • {quote['date']} — “{quote['text']}”"
                )

        lines.append("")  # spacing between reasons

    return "\n".join(lines)


In [ ]:
def compute_domain_scores_per_subject_from_zero_based_keys(
    sub_dpds,
    domains_csv_path,
    *,
    agg="mean"
):
    """
    Compute domain scores per subject from a dict like:
      key:  "{sub}-{i}" where i is 0-based (0..80) corresponding to dpds{i+1}
      value: list of scores across multiple days for that dpds item

    Returns:
      { sub : { domain : score } }

    Definition (MEAN OF MEANS):
      - For each subject and domain:
          1) compute the mean score for each dpds item in the domain (across days)
          2) aggregate those item-means across items (equal weight per item)
    """

    # --- Load domain -> list of dpds item numbers (1-based) ---
    df = pd.read_csv(domains_csv_path)

    domain_to_items = {}
    for _, row in df.iterrows():
        domain = row["Domain"]
        item_nums = [int(x) for x in re.findall(r"dpds(\d+)", str(row["Questions"]))]
        domain_to_items[domain] = item_nums  # keep list (not set)

    # --- Determine subjects present ---
    subjects = sorted({k.split("-")[0] for k in sub_dpds.keys()})
    subjects = [int(sub) for sub in subjects]

    out = {}
    for sub in subjects:
        out[sub] = {}
        for domain, item_nums in domain_to_items.items():

            per_item_means = []
            for dpds_num in item_nums:
                i = dpds_num - 1  # dpds1 -> key index 0
                vals = np.asarray(sub_dpds.get(f"{sub}-{i}", []), dtype=float)
                per_item_means.append(np.nanmean(vals))

            if len(per_item_means) == 0:
                out[sub][domain] = np.nan
            else:
                arr = np.asarray(per_item_means, dtype=float)
                if agg == "mean":
                    out[sub][domain] = np.nanmean(arr)
                elif agg == "median":
                    out[sub][domain] = float(np.median(arr))
                else:
                    raise ValueError("agg must be 'mean' or 'median'")

    return out

domain_scores = compute_domain_scores_per_subject_from_zero_based_keys(sub_dpds, './Data/domains_to_questions.csv')

In [ ]:
def find_close_high_low_subjects(df, domain, n=3, high_q=0.80, low_q=0.20):
    """
    For each domain:
      - High group: subjects with true >= high_q quantile
      - Low group:  subjects with true <= low_q quantile
    Within each group, return n subjects with smallest absolute error |llm - true|.

    Output tables include:
      subject, true, pred, abs_err
    """
    out = {}

    true_col = domain
    pred_col = f"{domain}_llm"

    d = df[["subject", true_col, pred_col]].copy()
    d = d.dropna(subset=[true_col, pred_col])

    d["abs_err"] = (d[pred_col] - d[true_col]).abs()

    hi_thr = d[true_col].quantile(high_q)
    lo_thr = d[true_col].quantile(low_q)

    hi = (
        d[d[true_col] >= hi_thr]
        .sort_values(["abs_err", true_col], ascending=[True, False])
        .head(n)
        .loc[:, ["subject", true_col, pred_col, "abs_err"]]
        .rename(columns={
            true_col: "true",
            pred_col: "pred"
        })
        .reset_index(drop=True)
    )

    lo = (
        d[d[true_col] <= lo_thr]
        .sort_values(["abs_err", true_col], ascending=[True, True])
        .head(n)
        .loc[:, ["subject", true_col, pred_col, "abs_err"]]
        .rename(columns={
            true_col: "true",
            pred_col: "pred"
        })
        .reset_index(drop=True)
    )

    out = {
        "high_true_close_pred": hi,
        "low_true_close_pred": lo
    }

    return out

In [ ]:
def write_domain_summaries_to_txt(
    summaries,
    out_path,
    *,
    section_separator="\n\n"
):
    """
    Write one or more formatted DPDS domain summaries to a readable .txt file.
    """

    # Normalize input
    if isinstance(summaries, str):
        summaries = [summaries]

    # Ensure directory exists (safe even if out_path has no dir)
    out_dir = os.path.dirname(out_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(section_separator.join(summaries))

    return out_path


# Run and Save Outputs

In [ ]:
#for domain in questions_list:
def work(domain):
    MODEL_TO_EVALUATE = gpt_5
    
    saved_out_results = pd.read_csv('./Data/domain_scores_per_subject_true_and_llm.csv')
    sub_list = find_close_high_low_subjects(saved_out_results, domain['domain'])
    domain_safe = domain['domain'].replace(" ", "_").lower()
    
    hi_res = []
    for subj in sub_list['high_true_close_pred'].subject.values.flatten():
        chat = OpenRouter(model_provider_order=MODEL_TO_EVALUATE)
        thoughts = '\n'.join(sub_transcripts[subj])
        message = get_prompt(thoughts, domain['domain'], domain['text'])
        schema = make_dpds_domain_json_schema(domain['domain'])
        required_values, cost = generic_messaging_wrapper(chat, system_role, message, schema)
        true_score = domain_scores[subj][domain['domain']]
        res = format_dpds_domain_summary(data=required_values, 
                                         domain=domain['domain'],
                                         subj=subj,
                                         true_score=true_score)
        hi_res.append(res)
    write_domain_summaries_to_txt(hi_res, out_path=f'./reasoning/hi_{domain_safe}.txt')
    
    lo_res = []   
    for subj in sub_list['low_true_close_pred'].subject.values.flatten():
        chat = OpenRouter(model_provider_order=MODEL_TO_EVALUATE)
        thoughts = '\n'.join(sub_transcripts[subj])
        message = get_prompt(thoughts, domain['domain'], domain['text'])
        schema = make_dpds_domain_json_schema(domain['domain'])
        required_values, cost = generic_messaging_wrapper(chat, system_role, message, schema)
        true_score = domain_scores[subj][domain['domain']]
        res = format_dpds_domain_summary(data=required_values, 
                                         domain=domain['domain'],
                                         subj=subj,
                                         true_score=true_score)
        lo_res.append(res)
    write_domain_summaries_to_txt(lo_res, out_path=f'./reasoning/lo_{domain_safe}.txt')

In [ ]:
procs = []
for domain in questions_list:
    proc = Process(target=work, args=(domain,))
    proc.start()
    procs.append(proc)
    
for proc in procs:
    proc.join()
procs = []